# Day 7: NumPy arrays, rock-fluid tables, and capillary pressure

**Time:** about 1 hour  |  **Needs:** Days 1 to 6
**Reservoir idea:** the laboratory rock-fluid table (relative permeability and capillary pressure against saturation), and what capillary pressure means for how oil and water sit in the reservoir
**Python you will learn:** NumPy arrays: creating them, arithmetic on a whole array at once, summaries (`sum`, `max`, `cumsum`, `argmax`), `np.linspace`, reading a table with `np.loadtxt`, interpolating with `np.interp`, plot styling

By the end of today you will be able to:

1. create NumPy arrays and calculate with whole arrays without writing a loop,
2. explain why simulators use arrays (speed and readability),
3. compute the Corey curves of Day 6 in a few lines and compare with your loop,
4. load a laboratory table from a text file in one line and interpolate it,
5. turn capillary pressure into a height above the free-water level.

> **Companion reading (optional, about 30 minutes).** The book *Python for Everybody* does not cover NumPy. Use the NumPy documentation page "NumPy: the absolute basics for beginners" (the sections on arrays, creating arrays, and basic array operations).

## 0. Setup and recap

Run the next cell.

**Recap of Day 6.** A dictionary of Corey parameters (`swc`, `sor`, `krw_max`, `kro_max`, `nw`, `no`) gave the curves $k_{rw}=k_{rw}^{\max}s^{n_w}$ and $k_{ro}=k_{ro}^{\max}(1-s)^{n_o}$, computed point by point in a loop and stored in lists of tuples.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

corey = {"swc": 0.2, "sor": 0.2, "krw_max": 0.4, "kro_max": 1.0, "nw": 2.0, "no": 2.0}
print("NumPy", np.__version__, "| ready")

## 1. Story: the loop that took all night

*(An invented but typical situation.)* A student writes a first simulator. It has 100 000 grid blocks and takes 1000 time steps, and in every step a Python loop updates every block, so there are 100 million updates. It runs for hours. A colleague rewrites the inner loop as one line on arrays and the same run finishes in minutes.

Nothing changed in the physics. The difference is that NumPy performs the loop **inside optimised compiled code**, instead of one slow Python step at a time. This is why almost every scientific Python program, including `kusim`, is written with arrays. Today we learn how.

## 2. Arrays

A NumPy **array** looks like a list of numbers but is a different kind of object: every item has the same type (usually decimal numbers), and arithmetic is applied to **all items at once**.

Predict what each of the last two lines prints, then run the cell. Be careful with the second one!

In [ ]:
as_list = [1, 2, 3]
as_array = np.array([1, 2, 3])

print(as_list * 2)
print(as_array * 2)
print(as_array + 10)
print(as_array * as_array)
print(np.sqrt(as_array))

With a **list**, `* 2` *repeats* it. With an **array**, `* 2` *multiplies every item*. Arrays work *element by element*: `a + b` adds item 0 to item 0, item 1 to item 1, and so on (the two arrays must have the same length). Functions like `np.exp`, `np.sqrt`, `np.log` also work on whole arrays.

### Making arrays

| You write | You get |
|---|---|
| `np.array([1500, 2000, 3500])` | an array from a list |
| `np.zeros(5)` / `np.ones(5)` | five 0s / five 1s |
| `np.linspace(0.2, 0.8, 7)` | 7 evenly spaced numbers from 0.2 to 0.8 **including both ends** |
| `np.arange(0, 10, 2)` | 0, 2, 4, 6, 8 (steps of 2, **excluding** the end) |

Useful facts about an array `a`: `len(a)` (number of items), `a.shape` (its size), `a[0]`, `a[-1]` (indexing works as for lists).

In [ ]:
sw = np.linspace(0.2, 0.8, 7)
print(sw)
print(len(sw), sw.shape)
print(np.arange(0, 10, 2))
print(np.zeros(3))

### Calculating with whole arrays

Here are the table pressures from Day 5 as an array. One line converts all of them from psi to kPa; there is no loop.

In [ ]:
PSI_TO_KPA = 6.894757

p_psi = np.array([1500, 2000, 3500, 4300])
p_kpa = p_psi * PSI_TO_KPA
print(p_kpa)

**Your turn 2.1.** On Day 4 you wrote a function that converts one pressure. Here the whole conversion back is needed: from the array `p_kpa_table = np.array([10000.0, 20000.0, 30000.0])` calculate the array `p_psi_table` (in psi) with **one line**.

In [ ]:
p_kpa_table = np.array([10000.0, 20000.0, 30000.0])

# YOUR CODE HERE
p_psi_table = None

In [ ]:
# Check (do not edit)
assert isinstance(p_psi_table, np.ndarray), "p_psi_table should be a NumPy array"
assert np.allclose(p_psi_table, [1450.377, 2900.754, 4351.131], atol=0.01), f"got {p_psi_table}"
print("Correct")

<details><summary>Solution</summary>

```python
p_psi_table = p_kpa_table / PSI_TO_KPA
```
</details>

**Your turn 2.2.** The simulator `kusim` stores the *reciprocal* $1/B_o$ of the formation volume factor (the equations use it). From `bo = np.array([1.522, 1.5, 1.478, 1.45])` calculate the array `inv_bo` of reciprocals.

In [ ]:
bo = np.array([1.522, 1.5, 1.478, 1.45])

# YOUR CODE HERE
inv_bo = None

In [ ]:
# Check (do not edit)
assert np.allclose(inv_bo, [0.657030, 0.666667, 0.676590, 0.689655], atol=1e-5), f"got {inv_bo}"
print("Correct:", np.round(inv_bo, 4))

<details><summary>Solution</summary>

```python
inv_bo = 1 / bo
```
</details>

### Loop or array: a race

Calculate $2.5x+1$ for a million numbers, with a loop and then with an array. (The time depends on your machine.)

In [ ]:
n = 1_000_000
numbers = list(range(n))

t0 = time.perf_counter()
result_loop = []
for v in numbers:
    result_loop.append(2.5 * v + 1)
t_loop = time.perf_counter() - t0

arr = np.arange(n)
t0 = time.perf_counter()
result_array = 2.5 * arr + 1
t_array = time.perf_counter() - t0

print(f"loop : {t_loop:.3f} s")
print(f"array: {t_array:.4f} s   ({t_loop / t_array:.0f} times faster)")

Arrays are usually tens of times faster for this kind of work. They are also **shorter and closer to the equation**: `2.5 * arr + 1` reads like the formula.

## 3. Summaries of an array

| You write | You get |
|---|---|
| `a.sum()`, `a.mean()` | total, average |
| `a.max()`, `a.min()` | largest, smallest |
| `a.argmax()`, `a.argmin()` | the **position** of the largest / smallest |
| `np.cumsum(a)` | the running totals, as an array |
| `np.abs(a)` | absolute value of every item |

On Day 3 you wrote loops for the total, the peak and its position, and a `cumulative()` function. Each is now one call:

In [ ]:
rates = np.array([2450, 2410, 2395, 0, 2360, 2330, 2310, 0, 2285, 2260])

print(rates.sum(), rates.mean(), rates.max(), rates.argmax())
print(np.cumsum(rates))

**Your turn 3.1.** For the array `rates` above store: `cum` (the running total, i.e. cumulative production, as an array), `peak_day` (the position of the highest rate) and `mean_rate` (the average over all 10 days, including the two zero days).

In [ ]:
# YOUR CODE HERE
cum = None
peak_day = None
mean_rate = None

In [ ]:
# Check (do not edit)
assert isinstance(cum, np.ndarray) and len(cum) == 10, "cum should be an array with 10 items"
assert cum[-1] == 18800, f"the last running total should be 18800, got {cum[-1]}"
assert peak_day == 0, f"the highest rate (2450) is at position 0, got {peak_day}"
assert mean_rate == 1880, f"mean_rate should be 1880, got {mean_rate}"
print("Correct")

<details><summary>Solution</summary>

```python
cum = np.cumsum(rates)
peak_day = rates.argmax()
mean_rate = rates.mean()
```
</details>

## 4. The Corey curves without a loop

On Day 6 you wrote `corey_krw(sw, props)` for one saturation, then a loop to build a table. With arrays, `sw` can be an array and the formulas apply to every point at once. Equations (1) and (2) of Day 6, repeated:

$$s = \frac{S_w - S_{wc}}{1 - S_{wc} - S_{or}}, \qquad k_{rw} = k_{rw}^{\max}\,s^{\,n_w}, \qquad k_{ro} = k_{ro}^{\max}\,(1-s)^{\,n_o}$$

**Your turn 4.1.** Write `corey_arrays(props, n)` returning three arrays `sw, krw, kro` of length `n`, with `sw` from `swc` to `1 - sor` (use `np.linspace`). No loop is needed. (We only use saturations inside the range, so no clamping is needed today; Day 8 adds it.)

In [ ]:
def corey_arrays(props, n):
    """Corey curves on n points from swc to 1 - sor. Returns arrays (sw, krw, kro)."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
sw, krw, kro = corey_arrays(corey, 7)
assert isinstance(sw, np.ndarray) and len(sw) == 7, "sw should be an array of length 7"
assert np.allclose(sw, [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]), f"sw is {sw}"
assert np.allclose(krw, [0, 0.011111, 0.044444, 0.1, 0.177778, 0.277778, 0.4], atol=1e-5), f"krw is {krw}"
assert np.allclose(kro, [1, 0.694444, 0.444444, 0.25, 0.111111, 0.027778, 0], atol=1e-5), f"kro is {kro}"
print("Correct: the same numbers as the Day 6 loop")

<details><summary>Solution</summary>

```python
def corey_arrays(props, n):
    """Corey curves on n points from swc to 1 - sor. Returns arrays (sw, krw, kro)."""
    sw = np.linspace(props["swc"], 1 - props["sor"], n)
    s = (sw - props["swc"]) / (1 - props["swc"] - props["sor"])
    krw = props["krw_max"] * s ** props["nw"]
    kro = props["kro_max"] * (1 - s) ** props["no"]
    return sw, krw, kro
```
Compare with Day 6: the equations are the same, but there is no loop, no `append`, and no tuple building.
</details>

**Your turn 4.2.** On Day 6 you found the crossover saturation with a list of tuples and `min`. With arrays it is `np.argmin(np.abs(krw - kro))`, the *position* of the smallest difference. Use 201 points and store `crossover_sw`.

In [ ]:
sw, krw, kro = corey_arrays(corey, 201)

# YOUR CODE HERE
crossover_sw = None

In [ ]:
# Check (do not edit)
assert abs(crossover_sw - 0.5676) < 0.005, f"crossover_sw should be about 0.5676, got {crossover_sw}"
print(f"Correct: Sw = {crossover_sw:.3f}, as on Day 6")

<details><summary>Solution</summary>

```python
i = np.argmin(np.abs(krw - kro))
crossover_sw = sw[i]
```
</details>

### Plot styling

A plot should be readable alone. The commands you need most:

| Command | Does |
|---|---|
| `plt.plot(x, y, "o-", color="tab:blue", linewidth=2, markersize=4, label="krw")` | the curve: marker, colour, thickness, legend text |
| `plt.title("...")`, `plt.xlabel("...")`, `plt.ylabel("...")` | titles and axis labels (**with units**) |
| `plt.xlim(0, 1)`, `plt.ylim(0, 1)` | axis range |
| `plt.legend()`, `plt.grid(True)` | legend and grid |
| `plt.figure(figsize=(6, 4))` | size of the figure (inches), put before the plot |

(Line style strings: `"-"` solid, `"--"` dashed, `":"` dotted; markers: `"o"` circle, `"s"` square.)

In [ ]:
sw, krw, kro = corey_arrays(corey, 101)

plt.figure(figsize=(6, 4))
plt.plot(sw, kro, color="tab:green", linewidth=2, label="oil, $k_{ro}$")
plt.plot(sw, krw, color="tab:blue", linewidth=2, label="water, $k_{rw}$")
plt.plot(crossover_sw, np.interp(crossover_sw, sw, krw), "ko", label="crossover")
plt.title("Corey relative permeability")
plt.xlabel("water saturation $S_w$ (fraction)")
plt.ylabel("relative permeability (fraction)")
plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

## 5. The laboratory table, loaded in one line

Corey curves are a convenient formula. A real study uses **laboratory measurements**, given as a table. The thesis (Application 1) uses this one: water saturation, oil and water relative permeability, and the oil-water capillary pressure in psi. The cell writes it to a file, as the data would arrive.

In [ ]:
sat_text = '''SW KRO KRW PCOW(PSI)
0.000 0.800 0.000 68.0
0.040 0.800 0.000 68.0
0.080 0.800 0.000 68.0
0.120 0.800 0.000 68.0
0.160 0.800 0.000 50.0
0.200 0.650 0.002 32.0
0.240 0.513 0.010 21.0
0.280 0.400 0.020 15.5
0.320 0.315 0.033 12.0
0.360 0.250 0.049 9.20
0.400 0.196 0.066 7.00
0.440 0.150 0.090 5.30
0.480 0.112 0.119 4.20
0.520 0.082 0.150 3.40
0.560 0.060 0.186 2.70
0.600 0.040 0.227 2.10
0.640 0.024 0.277 1.70
0.680 0.012 0.330 1.30
0.720 0.005 0.390 1.00
0.760 0.000 0.462 0.70
0.800 0.000 0.540 0.50
0.840 0.000 0.620 0.40
1.000 0.000 1.000 0.00
'''

with open("sat.txt", "w") as f:
    f.write(sat_text)
print("sat.txt written")

On Day 5 you wrote `read_table` to read such a file. NumPy can do it in **one line**:

```python
sw_t, kro_t, krw_t, pc_t = np.loadtxt("sat.txt", skiprows=1, unpack=True)
```

- `skiprows=1` skips the header line,
- `unpack=True` returns each **column** as its own array, so four names catch the four columns.

(`np.loadtxt` can also cut at commas with `delimiter=","`, and skips lines starting with `#`.)

**Your turn 5.1.** Load the file into `sw_t`, `kro_t`, `krw_t`, `pc_t`.

In [ ]:
# YOUR CODE HERE
sw_t = None
kro_t = None
krw_t = None
pc_t = None

In [ ]:
# Check (do not edit)
assert len(sw_t) == 23, f"the table has 23 rows, got {len(sw_t)}"
assert sw_t[-1] == 1.0 and krw_t[-1] == 1.0 and pc_t[-1] == 0.0, "the last row should be (1.0, 0.0, 1.0, 0.0)"
assert kro_t[0] == 0.8 and pc_t[0] == 68.0, "the first row should be (0.0, 0.8, 0.0, 68.0)"
print("Correct")

<details><summary>Solution</summary>

```python
sw_t, kro_t, krw_t, pc_t = np.loadtxt("sat.txt", skiprows=1, unpack=True)
```
</details>

Plot the table (we use the markers to show where the measurements are):

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(sw_t, kro_t, "o-", color="tab:green", markersize=4, label="oil, $k_{ro}$")
plt.plot(sw_t, krw_t, "o-", color="tab:blue", markersize=4, label="water, $k_{rw}$")
plt.title("Laboratory relative permeability (thesis Application 1)")
plt.xlabel("water saturation $S_w$ (fraction)")
plt.ylabel("relative permeability (fraction)")
plt.legend()
plt.grid(True)
plt.show()

Compare with the Corey curves: the thesis oil curve starts at $k_{ro}=0.8$ (not 1), it is exactly zero for $S_w \ge 0.76$ (so the table implies a residual oil saturation of 0.24, although the input file also lists a separate `SOR` of 0.2: real input data are not always consistent with themselves), and water keeps rising to 1 at $S_w=1$. Real data are never exactly a formula.

## 6. Interpolating a table with `np.interp`

You wrote `interp(x, xs, ys)` on Day 5 for one number. NumPy's `np.interp(x, xs, ys)` does the same, **and accepts an array of `x`**, returning an array. It is the function `kusim` uses.

In [ ]:
print(np.interp(0.5, sw_t, krw_t))                    # one value
sw_grid = np.linspace(0, 1, 11)
print(np.interp(sw_grid, sw_t, krw_t))                # eleven values at once

**Your turn 6.1.** Store in `krw_05` and `kro_05` the water and oil relative permeabilities at $S_w=0.5$ (interpolated from the table).

In [ ]:
# YOUR CODE HERE
krw_05 = None
kro_05 = None

In [ ]:
# Check (do not edit)
assert abs(krw_05 - 0.1345) < 1e-9, f"krw at 0.5 should be 0.1345 (between 0.119 and 0.150), got {krw_05}"
assert abs(kro_05 - 0.097) < 1e-9, f"kro at 0.5 should be 0.097 (between 0.112 and 0.082), got {kro_05}"
print("Correct")

<details><summary>Solution</summary>

```python
krw_05 = np.interp(0.5, sw_t, krw_t)
kro_05 = np.interp(0.5, sw_t, kro_t)
```
</details>

### Mobility and the mobility ratio

How easily a fluid moves depends on both the rock (relative permeability) and the fluid (viscosity). The **mobility** of a phase is $\lambda = k_r/\mu$. The **mobility ratio** compares the displacing fluid (water) with the displaced fluid (oil):

$$M = \frac{\lambda_w}{\lambda_o} = \frac{k_{rw}/\mu_w}{k_{ro}/\mu_o}$$

If $M>1$ the water moves more easily than the oil and tends to finger through it (a poor sweep); if $M<1$ the water pushes the oil like a piston (a good sweep). This is a key number in waterflooding, and the equation $\lambda = k_r/\mu$ appears in the simulator as `lam_o`, `lam_w`.

**Your turn 6.2.** Write `mobility_ratio(krw, kro, mu_w, mu_o)`, then store in `M_05` the ratio at $S_w=0.5$ for $\mu_w = 1.0$ cp and $\mu_o = 0.5$ cp.

In [ ]:
def mobility_ratio(krw, kro, mu_w, mu_o):
    """Water-oil mobility ratio."""
    # YOUR CODE HERE
    pass

M_05 = None

In [ ]:
# Check (do not edit)
M_05 = mobility_ratio(krw_05, kro_05, mu_w=1.0, mu_o=0.5)
assert abs(M_05 - 0.6933) < 1e-3, f"M at Sw = 0.5 should be about 0.693, got {M_05}"
assert abs(mobility_ratio(0.4, 1.0, 1.0, 1.0) - 0.4) < 1e-12, "with equal viscosities M = krw/kro"
print(f"Correct: M = {M_05:.3f} (below 1: favourable)")

<details><summary>Solution</summary>

```python
def mobility_ratio(krw, kro, mu_w, mu_o):
    """Water-oil mobility ratio."""
    return (krw / mu_w) / (kro / mu_o)

M_05 = mobility_ratio(krw_05, kro_05, mu_w=1.0, mu_o=0.5)
```
The function works for single numbers and for whole arrays, as it is built only from arithmetic.
</details>

## 7. Capillary pressure

The fourth column of the table, `PCOW`, is the **capillary pressure** between oil and water:

$$P_{cow} = P_o - P_w$$

In the tiny pores, the curved interface between oil and water carries a pressure jump, so the oil pressure is higher than the water pressure by $P_{cow}$. The jump is larger for smaller pores and for less water: in the table, $P_{cow}=68$ psi at $S_w=0$ and falls to zero at $S_w=1$.

### What it does in the reservoir

Capillary pressure decides **how oil and water sit before production starts**. Oil is lighter than water, so with height above the free-water level the oil pressure falls less steeply than the water pressure, and the difference $P_{cow}$ grows with height. The difference in pressure gradient is $0.433\,(\gamma_w-\gamma_o)$ psi/ft, where $\gamma$ are the fluid specific gravities (water 1.0; the thesis oil is 0.85). So the height $h$ above the free-water level where capillary pressure equals $P_{cow}$ is

$$h = \frac{P_{cow}}{0.433\,(\gamma_w - \gamma_o)} \tag{1}$$

Each row of the table therefore gives a saturation and the height at which it occurs: the **saturation-height curve**. Close to the free-water level the rock is mostly water; higher up the water saturation drops towards the connate value. (`0.433` is `PSI_PER_FT` in `kusim/constants.py`.)

**Your turn 7.1.** Write `height_above_fwl(pc, gamma_o=0.85, gamma_w=1.0)` using equation (1). It must work for a single number **and for a whole array** of capillary pressures. Then store in `height_t` the heights of the whole `pc_t` table.

In [ ]:
PSI_PER_FT = 0.433          # psi/ft per unit specific gravity

In [ ]:
def height_above_fwl(pc, gamma_o=0.85, gamma_w=1.0):
    """Height (ft) above the free-water level where the capillary pressure is pc (psi)."""
    # YOUR CODE HERE
    pass

height_t = None

In [ ]:
# Check (do not edit)
height_t = height_above_fwl(pc_t)
assert abs(height_above_fwl(32.0) - 492.7) < 0.1, f"32 psi should give about 492.7 ft, got {height_above_fwl(32.0)}"
assert abs(height_above_fwl(0.0)) < 1e-12, "zero capillary pressure is the free-water level itself (height 0)"
assert isinstance(height_t, np.ndarray) and len(height_t) == 23, "height_t should be an array with 23 items"
assert abs(height_t[0] - 68.0 / (0.433 * 0.15)) < 1e-9, "the first height is for 68 psi"
print(f"Correct: 68 psi corresponds to {height_t[0]:,.0f} ft above the free-water level")

<details><summary>Solution</summary>

```python
def height_above_fwl(pc, gamma_o=0.85, gamma_w=1.0):
    """Height (ft) above the free-water level where the capillary pressure is pc (psi)."""
    return pc / (PSI_PER_FT * (gamma_w - gamma_o))

height_t = height_above_fwl(pc_t)
```
</details>

Plot it the way petroleum engineers do: **water saturation across, height up**.

In [ ]:
plt.figure(figsize=(5, 4.5))
plt.plot(sw_t, height_t, "o-", markersize=4, color="tab:blue")
plt.title("Saturation-height curve")
plt.xlabel("water saturation $S_w$ (fraction)")
plt.ylabel("height above free-water level (ft)")
plt.xlim(0, 1)
plt.grid(True)
plt.show()

The curve shows a **transition zone**: water saturation climbs gradually from the connate value at the top of the transition up to 1 at the free-water level, rather than jumping at a sharp oil-water contact. How thick the zone is depends on the rock: a reservoir that is only 90 ft thick, like the thesis one, spans only a small part of this height axis.

## 8. Bridge: arrays inside `kusim`

Open `src/kusim/cases.py` and find `corey_rock_fluid`. Its body is your `corey_arrays`, written with `np.linspace` and array arithmetic:

```python
sw = np.linspace(swc, 1.0 - sor, points)
s = (sw - swc) / (1.0 - swc - sor)
return RockFluidTable(sw, kro_max * (1 - s) ** no, krw_max * s**nw, pc_max * (1 - s))
```
(It also builds a Pc table, going linearly from `pc_max` at $S_{wc}$ to 0 at $1-S_{or}$.) Then open `src/kusim/tables.py`: its `Table.lookup` is `np.interp`, which you used in Section 6, applied to a whole array of saturations (one per grid block) in one call.

And `src/kusim/model.py` has the `Equilibrium` class, which does Section 7 for every block: from its depth it gets the capillary pressure, and from the table it finds the water saturation. To read the table *backwards*, as saturation from pressure, `tables.py` has `invert_decreasing`, which reverses both arrays with `[::-1]` (reading an array backwards: Day 8's topic, slicing).

## 9. Self-quiz

Write short answers as comments, then open the answers.

1. `a = np.array([1, 2, 3])`: what are `a * 2`, `a + a`, `a.sum()`, `np.cumsum(a)` and `a.argmax()`?
2. What is the difference between `np.linspace(0, 1, 5)` and `np.arange(0, 1, 0.25)`?
3. Why is `2.5 * arr + 1` preferred to a loop for large arrays?
4. What does `unpack=True` do in `np.loadtxt`?
5. If $M = \lambda_w/\lambda_o$ is 3 in a waterflood, is that good or bad for sweep? Why?

In [ ]:
# 1.
# 2.
# 3.
# 4.
# 5.

<details><summary>Answers</summary>

1. `[2 4 6]`, `[2 4 6]`, `6`, `[1 3 6]` and `2` (the position of the largest item).
2. `linspace(0, 1, 5)` gives five numbers from 0 to 1 **including** 1: `0, 0.25, 0.5, 0.75, 1`. `arange(0, 1, 0.25)` gives numbers stepping by 0.25 **excluding** the end: `0, 0.25, 0.5, 0.75`. With `linspace` you choose the *count*; with `arange` the *step*.
3. The loop runs one Python step at a time; the array operation runs in compiled code on all items at once, so it is much faster, and it is shorter and reads like the equation.
4. It returns each column of the file as its own array instead of one 2-D array, so you can write `a, b, c = np.loadtxt(...)`.
5. Bad. $M>1$ means water is more mobile than oil, so it tends to finger through the oil and sweep it poorly.
</details>

## 10. Pocket reference (today's Python)

| You want to... | Write |
|---|---|
| an array from a list | `np.array([1, 2, 3])` |
| evenly spaced numbers (count) | `np.linspace(0, 1, 11)` |
| numbers by step | `np.arange(0, 10, 2)` |
| zeros / ones | `np.zeros(n)`, `np.ones(n)` |
| work on all items | `a * 2`, `a + b`, `a ** 2`, `np.exp(a)`, `1 / a` |
| summaries | `a.sum()`, `a.mean()`, `a.max()`, `a.min()` |
| position of max / min | `a.argmax()`, `a.argmin()` |
| running totals | `np.cumsum(a)` |
| size | `len(a)`, `a.shape` |
| load a table | `x, y = np.loadtxt("f.txt", skiprows=1, unpack=True)` |
| interpolate | `np.interp(x, xs, ys)` (x can be an array) |
| compare numbers | `np.allclose(a, b)` |
| style a plot | `plt.plot(x, y, "o-", color=..., label=...)`, `plt.title`, `plt.xlim`, `plt.figure(figsize=...)` |

## 11. Wrap-up

Today you:

- calculated with whole arrays and saw why simulators use them,
- reproduced the Day 6 Corey curves with no loop and found the crossover with `argmin`,
- loaded a laboratory table in one line and interpolated it,
- calculated the mobility ratio and read what it says about sweep,
- turned capillary pressure into a saturation-height curve, the start of the initial condition of a reservoir,
- saw the same arrays inside `kusim`.

**Next, Day 8:** more NumPy (slicing, masks that select items by a condition, `np.where`, `np.clip`) and the **fractional-flow curve**, which says what share of the flow is water at each saturation, the heart of the waterflood.